# 05 — Evaluation

Evaluate the trained YOLOv11 checkpoint on the held-out **test** split: precision/recall, mAP@0.5, mAP@0.5:0.95, confusion matrix, per-class PR curves, model-variant comparison, and a qualitative prediction grid.

In [1]:
# Install dependencies
%pip install -q "ultralytics==8.3.*" pandas numpy matplotlib seaborn pillow
# On a bare Linux box (no system libGL) OpenCV import can fail — if so, uncomment:
# !apt-get update -qq && apt-get install -y -qq libgl1
print("nb05 dependencies ready")

Note: you may need to restart the kernel to use updated packages.
nb05 dependencies ready


In [ ]:
%matplotlib inline

from IPython.display import display as ipy_display

# Imports
from ultralytics import YOLO
from pathlib import Path
import random
import pandas as pd, numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from PIL import Image

In [10]:
# Config — paths, classes, eval thresholds
DATA_YAML    = Path('../data/dataset/data.yaml').resolve()
WEIGHTS      = Path('../weights/best.pt').resolve()
TEST_IMG_DIR = Path('../data/dataset/images/test')

# Class order (must match notebook 02)
CLASSES = ['projector', 'whiteboard', 'fire_extinguisher', 'door_sign']

# Evaluation knobs
IMGSZ          = 640
CONF_THRESH    = 0.25
IOU_THRESH     = 0.5
N_QUALITATIVE  = 8
SEED           = 42

random.seed(SEED)
assert WEIGHTS.exists(), 'train the model in notebook 04 first'

## 1. Run validation on the test split

In [22]:
# ultralytics .val() — overall test-set metrics
# verbose=False suppresses noisy ANSI progress bars; plots=True saves PR_curve.png etc.
model = YOLO(WEIGHTS)
metrics = model.val(data=str(DATA_YAML), split='test', imgsz=IMGSZ,
                    conf=CONF_THRESH, iou=IOU_THRESH, verbose=False, plots=True)

print(f"{'Metric':<25} {'Value':>8}")
print("-" * 35)
print(f"{'mAP@0.5':<25} {metrics.box.map50:>8.4f}")
print(f"{'mAP@0.5:0.95':<25} {metrics.box.map:>8.4f}")
print(f"{'Precision (macro)':<25} {metrics.box.mp:>8.4f}")
print(f"{'Recall    (macro)':<25} {metrics.box.mr:>8.4f}")
print(f"\nResults saved to: {metrics.save_dir}")

Ultralytics 8.3.253  Python-3.13.12 torch-2.11.0+cu126 CUDA:0 (NVIDIA GeForce RTX 4060, 8188MiB)
YOLO11n summary (fused): 100 layers, 2,582,932 parameters, 0 gradients, 6.3 GFLOPs
val: Fast image access  (ping: 0.10.1 ms, read: 57.717.0 MB/s, size: 22.6 KB)
val: Scanning C:\Users\mitah\github_projects\ai_cv_project\data\dataset\labels\test.cache... 80 images, 7 backgrounds, 0 corrupt: 100% ━━━━━━━━━━━━ 80/80 3.1Mit/s 0.0s
                 Class     Images  Instances      Box(P          R      mAP50  mAP50-95): 0% ──────────── 0/5  4.7s


error: Caught error in DataLoader worker process 0.
Original Traceback (most recent call last):
  File "c:\Users\mitah\github_projects\ai_cv_project\.venv\Lib\site-packages\torch\utils\data\_utils\worker.py", line 358, in _worker_loop
    data = fetcher.fetch(index)  # type: ignore[possibly-undefined]
  File "c:\Users\mitah\github_projects\ai_cv_project\.venv\Lib\site-packages\torch\utils\data\_utils\fetch.py", line 54, in fetch
    data = [self.dataset[idx] for idx in possibly_batched_index]
            ~~~~~~~~~~~~^^^^^
  File "c:\Users\mitah\github_projects\ai_cv_project\.venv\Lib\site-packages\ultralytics\data\base.py", line 376, in __getitem__
    return self.transforms(self.get_image_and_label(index))
                           ~~~~~~~~~~~~~~~~~~~~~~~~^^^^^^^
  File "c:\Users\mitah\github_projects\ai_cv_project\.venv\Lib\site-packages\ultralytics\data\base.py", line 389, in get_image_and_label
    label["img"], label["ori_shape"], label["resized_shape"] = self.load_image(index)
 

## 2. Per-class metrics

In [13]:
# Per-class precision / recall / mAP table
p, r, ap50, ap = metrics.box.p, metrics.box.r, metrics.box.ap50, metrics.box.ap
per_class = pd.DataFrame({
    'class':       CLASSES,
    'precision':   np.round(p, 4),
    'recall':      np.round(r, 4),
    'mAP@0.5':     np.round(ap50, 4),
    'mAP@0.5:0.95': np.round(ap.mean(1) if ap.ndim > 1 else ap, 4),
})
per_class

,class,precision,recall,mAP@0.5,mAP@0.5:0.95
0,projector,1.0000,0.7974,0.9274,0.7538
1,whiteboard,0.9517,0.7391,0.8820,0.7917
2,fire_extinguisher,1.0000,0.9565,0.9780,0.8927
3,door_sign,0.9875,1.0000,0.9950,0.8755


## 3. Confusion matrix (counts + column-normalized)

In [ ]:
import io
from IPython.display import Image as IPImage, display as ipy_display

cm = metrics.confusion_matrix.matrix
labels = CLASSES + ['background']

fig, ax = plt.subplots(1, 2, figsize=(14, 5))
sns.heatmap(cm, annot=True, fmt='.0f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[0])
ax[0].set_title('Confusion matrix (counts)'); ax[0].set_xlabel('Predicted'); ax[0].set_ylabel('True')

cm_norm = cm / cm.sum(axis=0, keepdims=True).clip(min=1)
sns.heatmap(cm_norm, annot=True, fmt='.2f', xticklabels=labels, yticklabels=labels, cmap='Blues', ax=ax[1])
ax[1].set_title('Confusion matrix (column-normalized)'); ax[1].set_xlabel('Predicted'); ax[1].set_ylabel('True')
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
buf.seek(0)
plt.close(fig)
ipy_display(IPImage(data=buf.getvalue()))

## 4. Per-class PR curves
Ultralytics writes `PR_curve.png` under the val run directory.

In [ ]:
import io
from IPython.display import Image as IPImage, display as ipy_display

save_dir = Path(metrics.save_dir)
pr_img = save_dir / 'PR_curve.png'
if not pr_img.exists():
    candidates = sorted(save_dir.glob('*curve*.png')) + sorted(save_dir.glob('*PR*.png'))
    pr_img = candidates[0] if candidates else None

if pr_img and pr_img.exists():
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.imshow(Image.open(pr_img))
    ax.set_title('Per-class Precision-Recall curves')
    ax.axis('off')
    plt.tight_layout()
    buf = io.BytesIO()
    fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
    buf.seek(0)
    plt.close(fig)
    ipy_display(IPImage(data=buf.getvalue()))
else:
    print(f'No PR curve PNG found in {save_dir}')
    print('Available files:', [p.name for p in save_dir.glob('*.png')])

## 5. Model-variant comparison
Re-run cells 5–7 after training each variant (notebook 04 → change `CFG['model']`) and fill this table.

In [16]:
# Auto-detect the trained variant from parameter count, then populate the correct row
n_params_m = sum(p.numel() for p in model.model.parameters()) / 1e6
if n_params_m < 5:
    trained_variant = 'yolo11n'
elif n_params_m < 15:
    trained_variant = 'yolo11s'
else:
    trained_variant = 'yolo11m'

print(f"Detected variant: {trained_variant} ({n_params_m:.1f}M params)")

comparison = pd.DataFrame([
    {'variant': 'yolo11n', 'params_M': 2.6,  'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
    {'variant': 'yolo11s', 'params_M': 9.4,  'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
    {'variant': 'yolo11m', 'params_M': 20.0, 'mAP@0.5': None, 'mAP@0.5:0.95': None, 'latency_ms': None},
])
mask = comparison['variant'] == trained_variant
comparison.loc[mask, 'mAP@0.5']        = round(metrics.box.map50, 4)
comparison.loc[mask, 'mAP@0.5:0.95']   = round(metrics.box.map,   4)
comparison

Detected variant: yolo11n (2.6M params)


,variant,params_M,mAP@0.5,mAP@0.5:0.95,latency_ms
0,yolo11n,2.6,0.9456,0.8284,None
1,yolo11s,9.4,None,None,None
2,yolo11m,20.0,None,None,None


## 6. Qualitative prediction grid

In [ ]:
import io
from IPython.display import Image as IPImage, display as ipy_display

test_imgs = sorted(TEST_IMG_DIR.glob('*.jpg'))
sample = random.sample(test_imgs, min(N_QUALITATIVE, len(test_imgs)))
preds = model.predict(source=[str(p) for p in sample], imgsz=IMGSZ, conf=CONF_THRESH, verbose=False)

cols = 4
rows = (len(preds) + cols - 1) // cols
fig, axes = plt.subplots(rows, cols, figsize=(4 * cols, 4 * rows))
for ax, r in zip(axes.flatten(), preds):
    ax.imshow(r.plot()[:, :, ::-1])  # BGR -> RGB
    ax.set_title(Path(r.path).name, fontsize=8)
    ax.axis('off')
for ax in axes.flatten()[len(preds):]:
    ax.axis('off')
plt.tight_layout()

buf = io.BytesIO()
fig.savefig(buf, format='png', dpi=120, bbox_inches='tight')
buf.seek(0)
plt.close(fig)
ipy_display(IPImage(data=buf.getvalue()))

### Reporting checklist
- [ ] Per-class precision / recall / mAP table filled with real numbers
- [ ] Confusion matrix inspected — identify dominant confusion pairs
- [ ] PR curves attached
- [ ] Variant comparison table completed for `yolo11n`, `yolo11s`, `yolo11m`
- [ ] N_QUALITATIVE-image grid included in the report